In [1]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False


# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import make_scorer
from sklearn.metrics import classification_report
from sklearn.metrics import make_scorer, f1_score
from sklearn.metrics import confusion_matrix, classification_report, f1_score

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from xgboost import XGBClassifier

In [2]:
train = pd.read_csv("C:/Users/이민정/Desktop/부트캠프/파이널프로젝트/전처리/상위20개컬럼.csv")

In [3]:
X_train=train.drop(['Segment'],axis=1)
y=train['Segment']

In [4]:
# 문자열 클래스일 경우 숫자로 변환
le = LabelEncoder()
y = le.fit_transform(y)

In [5]:
# 데이터 분리하기
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2)
print(X_train.shape, X_val.shape, y_val.shape, y_train.shape)

(1920000, 20) (480000, 20) (480000,) (1920000,)


In [38]:
# 표준화하기 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # 학습용: fit + transform
X_val_scaled = scaler.transform(X_val) 

In [40]:
#학습
model = XGBClassifier(
    objective='multi:softmax',
    num_class=len(le.classes_),
    use_label_encoder=False,
    eval_metric='mlogloss',
    tree_method='gpu_hist'
)

model.fit(X_train, y_train)
y_val_train = model.predict(X_val)

In [42]:
# 교차검증
f1_micro = make_scorer(f1_score, average='micro')
scores = cross_val_score(model, X_train, y_train, cv=5, scoring=f1_micro)

In [43]:
print("교차검증 평균 F1:", scores.mean())

교차검증 평균 F1: 0.8687609374999999


In [17]:
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'n_estimators': [100, 300]
}

grid = GridSearchCV(
    estimator=XGBClassifier(objective='multi:softmax', num_class=5, use_label_encoder=False, eval_metric='mlogloss',
                            tree_method="hist", device="cuda"),
    param_grid=param_grid,
    scoring='f1_micro',
    cv=3
)

grid.fit(X_train, y_train)

print("최적 F1:", grid.best_score_)
print("최적 파라미터:", grid.best_params_)

[13:17:45] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "device" } are not used.

[13:18:11] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "device" } are not used.

[13:18:34] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "device" } are not used.

[13:18:57] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "device" } are not used.

[13:20:07] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "device" } are not used.

[13:21:15] WARNING: 

In [76]:
# 튜닝 결과 반영하여 최종 모델 정의
final_model = XGBClassifier(
    objective='multi:softmax',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    learning_rate=0.3,
    max_depth=7,
    n_estimators=300,
    use_label_encoder=False
)

In [78]:
# 전체 훈련 데이터로 학습
final_model.fit(X_train, y_train)

# 검증 데이터 성능 평가
y_val_pred = final_model.predict(X_val)
print("최종 F1 (micro):", f1_score(y_val, y_val_pred, average='micro'))
print(classification_report(y_val, y_val_pred, target_names=le.classes_))

최종 F1 (micro): 0.8727583333333334
              precision    recall  f1-score   support

           A       0.73      0.21      0.32       184
           B       0.33      0.03      0.06        30
           C       0.67      0.51      0.58     25401
           D       0.62      0.54      0.58     69403
           E       0.92      0.96      0.94    384982

    accuracy                           0.87    480000
   macro avg       0.65      0.45      0.50    480000
weighted avg       0.86      0.87      0.87    480000



In [79]:
best_model_path = 'model/best_model_classification.dat'
final_model.save_model(best_model_path)

In [80]:
import pickle

# 모델 저장
with open('final_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

# 레이블 인코더 저장
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# 스케일러 저장
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)